# Credit Default Scorecard: walkthrough

This notebook runs the full pipeline and walks through the results in order: features, binning, the points table, validation against the challenger, the cut-off decision and fairness monitoring.

Everything here is produced by the modules in `src/` and the SQL in `sql/`. Running `python run_pipeline.py` gives the same numbers.

In [1]:
import sys, json
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src import config
from src.build_features import build_feature_mart
from src.scorecard import build_models, reason_codes
from src.evaluate import summary, bootstrap_gini_diff, cutoff_table, risk_grades, fairness_table, psi

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 140)

## 1. Feature mart (built in DuckDB SQL)

The SQL layer turns the wide source file into one row per customer per month, then builds behavioural features with window functions. Data-quality checks run before anything is returned.

In [2]:
mart = build_feature_mart()
print(f"{len(mart):,} customers, default rate {mart[config.TARGET].mean():.1%}")
mart[config.MODEL_FEATURES].describe().T[["mean", "min", "50%", "max"]]

30,000 customers, default rate 22.1%


,mean,min,50%,max
log_credit_limit,11.663,9.210,11.849,13.816
status_latest,-0.017,-2.000,0.000,8.000
worst_status_6m,0.439,-2.000,0.000,8.000
worst_status_3m,0.248,-2.000,0.000,8.000
months_late_6m,0.834,0.000,0.000,6.000
current_late_streak,0.618,0.000,0.000,6.000
months_paid_in_full_6m,1.155,0.000,0.000,6.000
months_inactive_6m,0.814,0.000,0.000,6.000
utilisation_latest,0.424,-0.620,0.314,6.455
utilisation_avg_6m,0.373,-0.233,0.285,5.364


## 2. Train / hold-out split

Binning, feature selection and tuning use the 70% training sample only. The 30% hold-out is scored once at the end.

In [3]:
train, test = train_test_split(mart, test_size=config.TEST_SIZE, stratify=mart[config.TARGET],
                               random_state=config.SEED)
bins, selection_log, card, challenger, ch_info = build_models(train)
selection_log

,feature,iv,decision
0,status_latest,0.893,kept
1,months_late_6m,0.869,kept
2,worst_status_3m,0.826,dropped: WoE correlation > 0.75 with status_la...
3,worst_status_6m,0.791,dropped: WoE correlation > 0.75 with months_la...
4,current_late_streak,0.771,dropped: WoE correlation > 0.75 with status_la...
5,months_no_payment,0.430,kept
6,log_credit_limit,0.196,kept
7,repayment_ratio_avg,0.152,kept
8,utilisation_avg_6m,0.094,kept
9,balance_trend,0.078,kept


## 3. Weight-of-evidence bins for the strongest feature

Repayment status is kept as categories: −2 (no use), −1 (paid in full) and 0 (revolving) are not points on a scale.

In [4]:
bins["status_latest"].table

,feature,bin,n,share,bad,bad_rate,woe,iv
0,status_latest,<= -2,1933,0.092,244.000,0.126,0.675,0.034
1,status_latest,"(-2, -1]",3971,0.189,674.000,0.170,0.329,0.019
2,status_latest,"(-1, 0]",10285,0.490,1300.000,0.126,0.675,0.182
3,status_latest,"(0, 1]",2616,0.125,895.000,0.342,-0.605,0.053
4,status_latest,> 1,2195,0.105,1532.000,0.698,-2.095,0.606


## 4. The points table

600 points corresponds to good:bad odds of 50:1, and every 20 points doubles the odds.

In [5]:
card.points_table[["feature", "bin", "share", "bad_rate", "points"]]

,feature,bin,share,bad_rate,points
0,status_latest,<= -2,0.092,0.126,88
1,status_latest,"(-2, -1]",0.189,0.170,81
2,status_latest,"(-1, 0]",0.490,0.126,88
3,status_latest,"(0, 1]",0.125,0.342,63
4,status_latest,> 1,0.105,0.698,35
5,months_late_6m,<= 0,0.662,0.115,82
6,months_late_6m,"(0, 1]",0.148,0.299,71
7,months_late_6m,"(1, 2]",0.064,0.397,67
8,months_late_6m,> 2,0.126,0.602,60
9,months_no_payment,<= 0,0.752,0.155,78


## 5. Scorecard vs gradient-boosting challenger on the hold-out

In [6]:
y = test[config.TARGET].to_numpy()
s = card.score(test)
pd_card = card.pd_from_score(s)
pd_gbm = challenger.predict_proba(test[config.MODEL_FEATURES])[:, 1]
pd.DataFrame({"scorecard": summary(y, pd_card), "gradient boosting": summary(y, pd_gbm)}).T

,auc,gini,ks,brier
scorecard,0.745,0.490,0.401,0.140
gradient boosting,0.776,0.552,0.425,0.136


In [7]:
boot = bootstrap_gini_diff(y, pd_card, pd_gbm, n=config.N_BOOTSTRAP, seed=config.SEED)
print(f"Gini gain of challenger: {boot['diff_mean']:+.3f}  95% CI [{boot['diff_ci'][0]:.3f}, {boot['diff_ci'][1]:.3f}]")

Gini gain of challenger: +0.062  95% CI [0.047, 0.078]


## 6. What the extra Gini buys at the business decision

In [8]:
cut_card = cutoff_table(y, s)
cut_gbm = cutoff_table(y, -pd_gbm)
at = lambda t: t.loc[(t["approval_rate"] - 0.80).abs().idxmin()]
approved = round(0.80 * len(test))
for name, t in [("scorecard", cut_card), ("gradient boosting", cut_gbm)]:
    r = at(t)
    print(f"{name:18s} default rate among approved {r['bad_rate_approved']:.1%}  "
          f"-> {round(approved * r['bad_rate_approved']):,} defaults out of {approved:,} approved")

scorecard          default rate among approved 14.2%  -> 1,023 defaults out of 7,200 approved
gradient boosting  default rate among approved 13.7%  -> 986 defaults out of 7,200 approved


## 7. Risk grades and decline reasons

In [9]:
risk_grades(y, s, config.GRADE_CUTS)

,grade,customers,defaults,default_rate,min_score,max_score,share
0,A,4269,452,0.106,545.000,566.000,0.474
1,B,1936,314,0.162,530.000,544.000,0.215
2,C,1112,286,0.257,510.000,529.000,0.124
3,D,830,346,0.417,480.000,509.000,0.092
4,E,853,593,0.695,443.000,479.000,0.095


In [10]:
worst = test.iloc[int(np.argmin(s))]
print("Score:", int(s.min()))
for feature, lost in reason_codes(card, worst):
    print(f"  {feature}: {lost} points below the best bin")

Score: 443
  status_latest: 53 points below the best bin
  months_late_6m: 22 points below the best bin
  log_credit_limit: 20 points below the best bin


## 8. Stability and fairness monitoring

Sex, age and marital status are not model inputs. They are used here only to check outcomes at the 80% cut-off.

In [11]:
value, _ = psi(card.score(train), s)
print(f"Score PSI, train vs hold-out: {value:.4f}")
fairness_table(test, s, at(cut_card)["cutoff_score"])

Score PSI, train vs hold-out: 0.0012


,attribute,group,n,approval_rate,observed_default_rate,default_rate_if_approved,auc_within_group,approval_ratio_vs_highest
0,sex,female,5437,0.816,0.210,0.133,0.744,1.000
1,sex,male,3563,0.793,0.239,0.158,0.746,0.973
2,age_band,21-29,2882,0.770,0.224,0.135,0.762,0.922
3,age_band,30-39,3332,0.836,0.198,0.132,0.734,1.000
4,age_band,40-49,1958,0.817,0.239,0.160,0.746,0.978
5,age_band,50+,828,0.792,0.260,0.172,0.725,0.948
